# Section 1: LGD Exploration

**目标**: 从 Charged Off 贷款算出实际 LGD, 理解分布形态和主要驱动因素.

**产出**:
1. LGD 分布图 (找峰值, 判断单峰/双峰)
2. LGD by purpose / FICO / term 的均值表
3. 清洗好的 charged_off_lgd.csv, 给 Section 2 用

**要回答的 3 个问题**:
1. LGD 均值多少? (行业常识 70-85%)
2. 分布是单峰还是双峰? 决定 Section 2 建模策略
3. 哪些 segment LGD 明显偏高/偏低?

## 1.1 Imports & Paths

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# TODO: 改成你本地的路径
RAW_DIR = Path("data/raw/lendingclub")
ACCEPTED_FILE = RAW_DIR / "accepted_2007_to_2018Q4.csv.gz"

OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1.2 读原始数据, 只要 LGD 相关列

151 列全读太慢, 只读我们需要的. 核心是 recovery 4 列 + 基础信息.

In [ ]:
lgd_cols = [
    "id",
    "loan_amnt",
    "funded_amnt",
    "term",
    "int_rate",
    "issue_d",
    "loan_status",
    "purpose",
    "addr_state",
    "fico_range_low",
    "fico_range_high",
    "dti",
    "sub_grade",
    # 核心 recovery 字段
    "total_rec_prncp",
    "total_rec_int",
    "recoveries",
    "collection_recovery_fee",
    "out_prncp",
    "last_pymnt_amnt",
]

df_raw = pd.read_csv(
    ACCEPTED_FILE,
    compression="gzip",
    usecols=lgd_cols,
    low_memory=False,
)

print("Raw shape:", df_raw.shape)
df_raw["loan_status"].value_counts()

## 1.3 筛 Charged Off, 清洗字段

In [ ]:
co_df = df_raw[df_raw["loan_status"] == "Charged Off"].copy()
print(f"Charged Off loans: {len(co_df):,}")

# 字段清洗
co_df["issue_d"] = pd.to_datetime(co_df["issue_d"], format="%b-%Y", errors="coerce")
co_df["term"] = co_df["term"].astype(str).str.extract(r"(\d+)").astype(float)
co_df["fico_avg"] = (co_df["fico_range_low"] + co_df["fico_range_high"]) / 2

# 看看 recovery 列的分布
co_df[["funded_amnt", "total_rec_prncp", "recoveries", "collection_recovery_fee"]].describe()

## 1.4 计算 LGD

```
principal_recovered = total_rec_prncp + recoveries - collection_recovery_fee
LGD = 1 - principal_recovered / funded_amnt
```

注意: 少数极端值可能超出 [0, 1], 我们 clip 一下.

In [ ]:
co_df["principal_recovered"] = (
    co_df["total_rec_prncp"].fillna(0)
    + co_df["recoveries"].fillna(0)
    - co_df["collection_recovery_fee"].fillna(0)
)
co_df["LGD"] = 1 - (co_df["principal_recovered"] / co_df["funded_amnt"])
co_df["LGD"] = co_df["LGD"].clip(0, 1)

print("LGD summary:")
print(co_df["LGD"].describe())

## 1.5 LGD 分布图 (最重要的一张图)

看峰在哪里, 是否双峰.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(co_df["LGD"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("LGD")
axes[0].set_ylabel("Number of Loans")
axes[0].set_title("LGD Distribution (Charged Off Loans)")
axes[0].axvline(co_df["LGD"].mean(), color="red", linestyle="--",
                label=f"Mean = {co_df['LGD'].mean():.2f}")
axes[0].axvline(co_df["LGD"].median(), color="green", linestyle="--",
                label=f"Median = {co_df['LGD'].median():.2f}")
axes[0].legend()

# Boxplot (参考)
axes[1].boxplot(co_df["LGD"].dropna(), vert=False)
axes[1].set_xlabel("LGD")
axes[1].set_title("LGD Boxplot")

plt.tight_layout()
plt.show()

## 1.6 LGD by Purpose

In [ ]:
lgd_by_purpose = (
    co_df.groupby("purpose")["LGD"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)
print(lgd_by_purpose)

lgd_by_purpose["mean"].plot(kind="barh", figsize=(10, 6))
plt.xlabel("Mean LGD")
plt.title("Mean LGD by Loan Purpose")
plt.tight_layout()
plt.show()

## 1.7 LGD by FICO Band

In [ ]:
co_df["fico_band"] = pd.cut(
    co_df["fico_avg"],
    bins=[0, 640, 680, 720, 760, 850],
    labels=["<640", "640-680", "680-720", "720-760", "760+"]
)

lgd_by_fico = co_df.groupby("fico_band")["LGD"].agg(["mean", "median", "count"])
print(lgd_by_fico)

lgd_by_fico["mean"].plot(kind="bar", figsize=(8, 5), color="steelblue")
plt.ylabel("Mean LGD")
plt.title("Mean LGD by FICO Band")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 1.8 LGD by Term

In [ ]:
lgd_by_term = co_df.groupby("term")["LGD"].agg(["mean", "median", "count"])
print(lgd_by_term)

## 1.9 保存清洗好的 Charged Off 数据

Section 2 直接读这个文件, 不用再重新走清洗流程.

In [ ]:
co_df.to_csv(OUT_DIR / "charged_off_lgd.csv", index=False)
print(f"Saved to {OUT_DIR / 'charged_off_lgd.csv'}")
print(f"Rows: {len(co_df):,}")

---

## Section 1 结论 (填写)

跑完上面后填写这三个观察:

1. **LGD 均值 / 中位数**: ___ / ___
2. **分布形态**: 单峰 / 双峰, 峰值在 ___
3. **LGD 最高的 purpose**: ___ ; **LGD 最低的 purpose**: ___
4. **FICO 对 LGD 有影响吗**: (通常影响不大, 因为违约后回收跟 FICO 关系弱)
5. **60 月 vs 36 月 LGD 差异**: ___